[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# A Complete Workflow &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]


def raw_export(network, seed=7):
    """The network as a station register exports it: some private stations' elevations missing,
    terrain typed by hand in several ways, and a few owners left blank."""
    rng = np.random.default_rng(seed)
    raw = network.copy()
    raw["elevation_m"] = raw["elevation_m"].astype(float)
    hidden = (raw["owner"] == "private").to_numpy() & (rng.uniform(size=len(raw)) < 0.3)
    raw.loc[hidden, "elevation_m"] = np.nan
    retyped = rng.uniform(size=len(raw)) < 0.08
    raw.loc[retyped, "terrain"] = raw.loc[retyped, "terrain"].str.upper() + " "
    blank = rng.uniform(size=len(raw)) < 0.02
    raw.loc[blank, "owner"] = np.nan
    return raw


raw = raw_export(network)

print("scikit-learn", sklearn.__version__, "|", len(raw), "rows in the raw export")

scikit-learn 1.6.1 | 1200 rows in the raw export


The answers share the worked examples' cleaning, preparation, split and folds, made again here since this notebook runs on its own:

In [2]:
def clean_text(table):
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


def prepare():
    numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    text = make_pipeline(SimpleImputer(strategy="most_frequent"),
                         OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("numbers", numbers, NUMERIC), ("text", text, TEXT)])


NUMERIC = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
TEXT = ["terrain", "owner"]
clean = clean_text(raw)
X_train, X_test, y_train, y_test = train_test_split(clean[NUMERIC + TEXT], clean["jan_mean"],
                                                    test_size=0.2, random_state=42)
folds = KFold(5, shuffle=True, random_state=0)

**1.** Add `RandomForestRegressor(min_samples_leaf=5, random_state=0)` to the candidates as `"forest, leaf 5"`, and print the cross-validated mean error of all four on the training rows.

In [3]:
candidates = {
    "mean only": DummyRegressor(),
    "linear": LinearRegression(),
    "forest": RandomForestRegressor(random_state=0),
    "forest, leaf 5": RandomForestRegressor(min_samples_leaf=5, random_state=0),
}
for name, estimator in candidates.items():
    model = Pipeline([("prepare", prepare()), ("model", estimator)])
    scores = cross_validate(model, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error")
    print(f"{name:15} {-scores['test_score'].mean():.3f}")

mean only       2.350
linear          0.753
forest          0.799
forest, leaf 5  0.809


Leaves of at least five stations make each tree smoother, and the forest still trails the linear model, as it did in the search.

**2.** Print the chosen model's mean absolute error on the test rows for each `owner`, with the number of test stations for each.

In [4]:
chosen = Pipeline([("prepare", prepare()), ("model", LinearRegression())]).fit(X_train, y_train)
errors = (chosen.predict(X_test) - y_test).abs()

errors.groupby(X_test["owner"]).agg(["count", "mean"]).round(2)

,count,mean
owner,,
airport,17,0.91
met,92,0.80
private,41,0.95
road,83,0.75


Grouping a Series by another column of the same index lines the errors up with each station's owner. The test stations with no owner on record are left out, since `groupby` drops missing keys.

**3.** Predict the January mean of a planned mountain station at latitude 62.0, longitude 9.5, elevation 1,100 m, 140 km from the coast, installed in 2026, owned by `road`, with its terrain typed `MOUNTAIN `, using `chosen` and `clean_text`.

In [5]:
chosen = Pipeline([("prepare", prepare()), ("model", LinearRegression())]).fit(X_train, y_train)
planned = pd.DataFrame([{
    "latitude": 62.0, "longitude": 9.5, "elevation_m": 1100, "coast_km": 140.0,
    "installed": 2026, "terrain": "MOUNTAIN ", "owner": "road",
}])

print(chosen.predict(clean_text(planned)).round(1))

[-14.2]


`clean_text` turns `MOUNTAIN ` into `mountain`, a value the encoder knows. Without it the terrain is read as unknown and ignored, as Common errors in the **A Complete Workflow** notebook shows.

**4.** Fit the chosen pipeline without `elevation_m` among the numbers, and print its cross-validated mean error on the training rows beside the full model's, to see what the elevation is worth.

In [6]:
def prepare_without_elevation():
    numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    text = make_pipeline(SimpleImputer(strategy="most_frequent"),
                         OneHotEncoder(handle_unknown="ignore"))
    kept = [name for name in NUMERIC if name != "elevation_m"]
    return ColumnTransformer([("numbers", numbers, kept), ("text", text, TEXT)])


for name, prep in [("with elevation", prepare()), ("without", prepare_without_elevation())]:
    model = Pipeline([("prepare", prep), ("model", LinearRegression())])
    scores = cross_validate(model, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error")
    print(f"{name:15} {-scores['test_score'].mean():.3f}")

with elevation  0.753
without         0.983


The columns a `ColumnTransformer` leaves unnamed are dropped by default, so leaving `elevation_m` out of the list removes it from the model. The error grows from 0.753 to 0.983 degrees without it, which is what the elevation is worth to this model.

---

&#8592; **Back to:** [A Complete Workflow](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/14-a-complete-workflow.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
